# Turn-taking and barge-in
Complete [setup](../../instructions/00_setup.md) and quickstart first. This **live Azure** exercise streams microphone input and plays response audio on the kernel machine for 25 seconds. Use a headset; speakers can cause echo-triggered interruptions. No audio is saved.

Ask: **Explain the water cycle in detail.** During playback, say: **Stop. Give me a one-sentence summary instead.** Repeat with a short mid-sentence pause to compare boundary behavior.

In [ ]:
from pathlib import Path
import sys
from dotenv import load_dotenv

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/shared/realtime.py').is_file())
sys.path.insert(0, str(root / 'src'))
from shared.realtime import Settings, connect, converse
load_dotenv(root / '.env', override=False)
settings = Settings.from_env()
_ = settings.url
print('Configuration validated locally. No live result yet.')

In [ ]:
silence_ms = 500  # Re-run with 1000 to compare pause tolerance, using a fresh session.
async with connect(settings, 'Explain science clearly. Follow requests to stop or summarize.', silence_ms=silence_ms) as session:
    print('Live microphone and playback starting for 25 seconds. Speak now.')
    events = await converse(session, seconds=25)
event_types = {event['type'] for event in events}
if 'response.output_audio.delta' not in event_types:
    raise RuntimeError('No response audio received; this is not a successful voice exercise.')
events

## Observe
Look for `input_audio_buffer.speech_started`, `input_audio_buffer.speech_stopped`, `response.done`, and `conversation.item.truncated`. Confirm by listening that the old playback stops and the new request receives a response. Event presence alone is not proof of good interruption quality.

Server VAD automatically cancels generation; the client discards queued audio and truncates assistant history at the PCM submitted to the audio device. Already submitted device buffers cannot be recalled, so a short audio tail can remain and this boundary is approximate. This notebook does not implement echo cancellation or promise production-grade latency. It does not enable semantic VAD.

Re-run with `silence_ms = 1000`; note responsiveness versus premature end-of-turn. Interrupt the kernel to stop early; the streams and connection are closed. Continue to [multilingual](03_multilingual_code_switching.ipynb).